# Agentic Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local **agentic workflow** for English-to-Chinese cross-lingual dialogue summarization.

The proposed pipeline is:

```text
English Dialogue
→ Agent 1: English Summarization
→ Agent 2: Chinese Translation
→ Agent 3: Evaluation-Revision Agent
→ Final Chinese Summary
```

The agents are controlled in this notebook. Ollama only serves the local models.


## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the models locally.

### Recommended model setup

```bash
ollama pull qwen3.5:9b
ollama pull translategemma:12b
```

If `translategemma:12b` is too slow on your machine, use:

```bash
ollama pull translategemma:4b
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages
# Run this only once if the packages are not installed.

!pip install requests pandas tqdm



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [18]:
# current working path check
import os
from pathlib import Path

PROJECT_ROOT = Path("/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization")
os.chdir(PROJECT_ROOT)

print("Current working directory:", Path.cwd())

Current working directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization


In [20]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# Agent models
SUMMARIZATION_MODEL = "qwen3.5:9b"
TRANSLATION_MODEL = "translategemma:12b"
EVALUATION_REVISION_MODEL = "qwen3.5:9b"

# If translategemma:12b is too slow, use:
# TRANSLATION_MODEL = "translategemma:4b"

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# Input dataset path
RAW_TEST_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files
FULL_OUTPUT_PATH = OUTPUT_DIR / "three_agent_outputs.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "three_agent_final_summaries.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "three_agent_errors.jsonl"

print("Raw test path:", RAW_TEST_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Raw test path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/raw/test.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/three_agent_outputs.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/three_agent_final_summaries.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/three_agent_errors.jsonl


In [21]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['translategemma:12b', 'qwen3.5:9b']


In [22]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [ ]:
# Cell 5: Prompt templates

SUMMARIZATION_PROMPT = """You are an English dialogue summarization agent.

Your task is to summarize the given English dialogue into a concise English summary.

Requirements:
- Preserve the main intent, key events, decisions, and important entities.
- Prioritize the final outcome of the dialogue over earlier tentative plans.
- If a plan changes during the dialogue, summarize both the original plan and the final revised plan only when necessary.
- Clearly distinguish between an initial plan, a cancellation, and a final agreement.
- Focus only on information that is important for the final dialogue summary.
- Do not include side comments, greetings, filler expressions, or travel-related small talk unless they directly affect the main event.
- Be careful with temporal expressions. Distinguish past events, current situations, and future plans.
- If the timing of an event is unclear, do not guess whether it is past or future.
- Do not infer unstated objects, causes, or meanings from figurative language.
- Do not turn quoted speech or jokes into literal facts.
- Do not add information that is not explicitly supported by the dialogue.
- Do not include minor details unless they are necessary for understanding the main point.
- Keep the summary concise, preferably two sentences.
- Use clear and natural English.
- Output only the English summary.

Input dialogue:
{dialogue}

English summary:
"""


TRANSLATION_PROMPT = """You are a professional English-to-Simplified-Chinese translation agent.

Your task is to translate the given English summary into natural Simplified Chinese.

Requirements:
- Preserve the original meaning accurately.
- Preserve names, numbers, dates, and important entities.
- Use Chinese transliterations for common English names.
- Do not include the original English names in parentheses unless necessary.
- Do not add new information.
- Do not remove important information.
- Use fluent and natural Simplified Chinese.
- Output only the Chinese translation.

English summary:
{english_summary}

Chinese summary:
"""


EVALUATION_REVISION_PROMPT = """You are a Chinese summary evaluation and revision agent.

Your task is to evaluate and revise a Chinese summary for an English dialogue.

You will receive:
1. The original English dialogue
2. An intermediate English summary
3. An initial Chinese summary generated by a translation agent

Your goal is to produce the final Chinese summary.

Check the following:
- Semantic faithfulness: The final Chinese summary must be supported by the original dialogue.
- Coverage: It should preserve the main events, decisions, intentions, and important entities.
- Temporal consistency: It must correctly represent past, present, and future events.
- Entity consistency: It must preserve names, people, and relationships accurately.
- Pragmatic interpretation: Do not over-interpret jokes, figurative expressions, or side comments.
- Cultural and linguistic naturalness: The Chinese should be fluent, concise, and natural.
- Conciseness: Remove unnecessary side details unless they affect the main event.

Important rules:
- Do not add information that is not supported by the dialogue.
- If the Chinese summary includes incorrect or unnecessary details, revise it.
- Do not include explanations.
- Output only the final revised Simplified Chinese summary.

Original English Dialogue:
{dialogue}

Intermediate English Summary:
{english_summary}

Initial Chinese Summary:
{chinese_summary}

Final revised Chinese summary:
"""

In [24]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [25]:
# Cell 7: Agent functions

def summarization_agent(dialogue: str) -> str:
    """Agent 1: Generate an intermediate English summary."""
    prompt = SUMMARIZATION_PROMPT.format(dialogue=dialogue)

    return call_ollama(
        model=SUMMARIZATION_MODEL,
        prompt=prompt,
        temperature=0.2,
    )


def translation_agent(english_summary: str) -> str:
    """Agent 2: Translate the English summary into Simplified Chinese."""
    prompt = TRANSLATION_PROMPT.format(english_summary=english_summary)

    return call_ollama(
        model=TRANSLATION_MODEL,
        prompt=prompt,
        temperature=0.1,
    )


def evaluation_revision_agent(
    dialogue: str,
    english_summary: str,
    chinese_summary: str,
) -> str:
    """Agent 3: Evaluate and revise the Chinese summary."""
    prompt = EVALUATION_REVISION_PROMPT.format(
        dialogue=dialogue,
        english_summary=english_summary,
        chinese_summary=chinese_summary,
    )

    return call_ollama(
        model=EVALUATION_REVISION_MODEL,
        prompt=prompt,
        temperature=0.1,
    )

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [26]:
# Cell 8: Three-agent pipeline

def run_three_agent_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    """Run the full three-agent pipeline on one example."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    # Agent 1: English summarization
    english_summary = summarization_agent(dialogue)

    # Agent 2: Chinese translation
    initial_chinese_summary = translation_agent(english_summary)

    # Agent 3: Evaluation and revision
    final_chinese_summary = evaluation_revision_agent(
        dialogue=dialogue,
        english_summary=english_summary,
        chinese_summary=initial_chinese_summary,
    )

    return {
        "id": sample_id,
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,
        "agent1_english_summary": english_summary,
        "agent2_initial_chinese_summary": initial_chinese_summary,
        "agent3_final_chinese_summary": final_chinese_summary,
        "final_summary": final_chinese_summary,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [27]:
# Cell 9: Load top examples from the original test.json file

def load_top_examples_from_test_json(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the top n examples from the original JSON test file."""
    if not path.exists():
        raise FileNotFoundError(f"Dataset not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"test_{i+1:05d}",
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_top_examples_from_test_json(RAW_TEST_PATH, n=5)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 5 examples.
First example:
{'id': 'test_00001', 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye", 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.", 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。'}


In [28]:
# Cell 10: Run the three-agent pipeline for the first example from test.json

result = run_three_agent_pipeline(test_data[0])
result

{'id': 'test_00001',
 'dialogue': "Hannah: Hey, do you have Betty's number?\nAmanda: Lemme check\nHannah: <file_gif>\nAmanda: Sorry, can't find it.\nAmanda: Ask Larry\nAmanda: He called her last time we were at the park together\nHannah: I don't know him well\nHannah: <file_gif>\nAmanda: Don't be shy, he's very nice\nHannah: If you say so..\nHannah: I'd rather you texted him\nAmanda: Just text him 🙂\nHannah: Urgh.. Alright\nHannah: Bye\nAmanda: Bye bye",
 'reference_english_summary': "Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.",
 'reference_chinese_summary': '汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。',
 'agent1_english_summary': "Amanda could not find Betty's number and suggested Hannah ask Larry, who called her recently, though Hannah prefers Amanda to contact Larry directly.",
 'agent2_initial_chinese_summary': '艾曼达找不到贝蒂的电话号码，建议汉娜让拉里帮忙联系，因为拉里最近给她打电话过。不过，汉娜更希望艾曼达直接联系拉里。',
 'agent3_final_chinese_summary': '艾曼达没找到贝蒂的号码，建议汉娜找拉里，因为拉里上次在公园时曾联系过她。汉娜不太认识拉里，但艾曼达鼓励她别害

In [29]:
# Cell 11: Print three-agent pipeline result clearly

def print_three_agent_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Agent 1: English Summary ===")
    print(result["agent1_english_summary"])
    print()

    print("=== Agent 2: Initial Chinese Summary ===")
    print(result["agent2_initial_chinese_summary"])
    print()

    print("=== Agent 3: Final Revised Chinese Summary ===")
    print(result["agent3_final_chinese_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Final Chinese Summary ===")
    print(result["final_summary"])


print_three_agent_result(result)

=== Original Dialogue ===
Hannah: Hey, do you have Betty's number?
Amanda: Lemme check
Hannah: <file_gif>
Amanda: Sorry, can't find it.
Amanda: Ask Larry
Amanda: He called her last time we were at the park together
Hannah: I don't know him well
Hannah: <file_gif>
Amanda: Don't be shy, he's very nice
Hannah: If you say so..
Hannah: I'd rather you texted him
Amanda: Just text him 🙂
Hannah: Urgh.. Alright
Hannah: Bye
Amanda: Bye bye

=== Agent 1: English Summary ===
Amanda could not find Betty's number and suggested Hannah ask Larry, who called her recently, though Hannah prefers Amanda to contact Larry directly.

=== Agent 2: Initial Chinese Summary ===
艾曼达找不到贝蒂的电话号码，建议汉娜让拉里帮忙联系，因为拉里最近给她打电话过。不过，汉娜更希望艾曼达直接联系拉里。

=== Agent 3: Final Revised Chinese Summary ===
艾曼达没找到贝蒂的号码，建议汉娜找拉里，因为拉里上次在公园时曾联系过她。汉娜不太认识拉里，但艾曼达鼓励她别害羞，汉娜仍希望艾曼达直接给拉里发信息，最终艾曼达同意并道别。

=== Reference English Summary ===
Hannah needs Betty's number but Amanda doesn't have it. She needs to contact Larry.

=== Reference Chinese Summary

## 4. Save Results

This saves all intermediate outputs and the final output.


In [30]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/three_agent_outputs.jsonl
CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/three_agent_final_summaries.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/three_agent_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [31]:
# Cell 13: Batch inference with three-agent pipeline

MAX_EXAMPLES = 5
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running three-agent pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        continue

    try:
        record = run_three_agent_pipeline(ex)
        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running three-agent pipeline:   0%|          | 0/5 [00:00<?, ?it/s]

Finished. Outputs saved to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/three_agent_outputs.jsonl


## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [32]:
# Cell 14: Export final summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "dialogue": record.get("dialogue", ""),
        "final_summary": record.get("final_summary", ""),
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),
        "agent1_english_summary": record.get("agent1_english_summary", ""),
        "agent2_initial_chinese_summary": record.get("agent2_initial_chinese_summary", ""),
        "agent3_final_chinese_summary": record.get("agent3_final_chinese_summary", ""),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/three_agent_final_summaries.csv


,id,dialogue,final_summary,reference_english_summary,reference_chinese_summary,agent1_english_summary,agent2_initial_chinese_summary,agent3_final_chinese_summary
0,test_00001,"Hannah: Hey, do you have Betty's number?\nAman...",艾曼达没找到贝蒂的电话号码，建议找拉里，因为拉里上次在公园时联系过她。汉娜不太熟悉拉里，起初...,Hannah needs Betty's number but Amanda doesn't...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。,Amanda could not find Betty's number and sugge...,艾曼达找不到贝蒂的电话号码，建议问拉里。汉娜虽然不太情愿，但最终同意让艾曼达给拉里发短信。,艾曼达没找到贝蒂的电话号码，建议找拉里，因为拉里上次在公园时联系过她。汉娜不太熟悉拉里，起初...
1,test_00002,Eric: MACHINE!\r\nRob: That's so gr8!\r\nEric:...,埃里克和罗布讨论喜剧演员 Machine 的脱口秀，埃里克尤其喜欢其中的火车片段。罗布在 Y...,Eric and Rob are going to watch a stand-up on ...,埃里克和罗伯要在youtube上看一场单口相声。,Eric and Rob discuss a stand-up comedy perform...,埃里克和罗布讨论了一位名为“Machine”的喜剧演员的脱口秀表演，埃里克特别喜欢其中关于火...,埃里克和罗布讨论喜剧演员 Machine 的脱口秀，埃里克尤其喜欢其中的火车片段。罗布在 Y...
2,test_00003,"Lenny: Babe, can you help me with something?\r...",莱尼在鲍勃建议优先选择搭配性更好的款式后，决定购买第一或第三条裤子，并同意以质量为首要考虑因素。,Lenny can't decide which trousers to buy. Bob ...,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。,Lenny decides to purchase either the first or ...,莱尼决定购买第一件或第三件裤子，他更注重质量而不是款式多样性。鲍勃建议他选择第一件，因为那样...,莱尼在鲍勃建议优先选择搭配性更好的款式后，决定购买第一或第三条裤子，并同意以质量为首要考虑因素。
3,test_00004,"Will: hey babe, what do you want for dinner to...",艾玛今晚不饿，不想做饭，但很快就会到家，到家后会通知威尔。,Emma will be home soon and she will let Will k...,艾玛很快就会回家，而且她会告诉威尔。,Emma is not hungry and will not be cooking din...,艾玛现在不饿，所以她不会做晚饭。不过她很快就会到家，到时候她会通知威尔。,艾玛今晚不饿，不想做饭，但很快就会到家，到家后会通知威尔。
4,test_00005,"Ollie: Hi , are you in Warsaw\r\nJane: yes, ju...",简和奥利确认周五下午 6 点课后一起吃午饭，此前两人还讨论了 18 号的派对，并就奥利手机里...,Jane is in Warsaw. Ollie and Jane has a party....,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...,Jane and Ollie confirm a lunch meeting for Fri...,简和奥利确认周五晚上6点一起吃午饭，这是在他们讨论了18号的派对以及关于一瓶失踪的威士忌的有...,简和奥利确认周五下午 6 点课后一起吃午饭，此前两人还讨论了 18 号的派对，并就奥利手机里...


In [33]:
# Cell 15: Compare Agent 2 initial summary and Agent 3 final summary

comparison_columns = [
    "id",
    "agent2_initial_chinese_summary",
    "agent3_final_chinese_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,agent2_initial_chinese_summary,agent3_final_chinese_summary,reference_chinese_summary
0,test_00001,艾曼达找不到贝蒂的电话号码，建议问拉里。汉娜虽然不太情愿，但最终同意让艾曼达给拉里发短信。,艾曼达没找到贝蒂的电话号码，建议找拉里，因为拉里上次在公园时联系过她。汉娜不太熟悉拉里，起初...,汉娜需要贝蒂的电话号码，但阿曼达没有。她得联系拉里。
1,test_00002,埃里克和罗布讨论了一位名为“Machine”的喜剧演员的脱口秀表演，埃里克特别喜欢其中关于火...,埃里克和罗布讨论喜剧演员 Machine 的脱口秀，埃里克尤其喜欢其中的火车片段。罗布在 Y...,埃里克和罗伯要在youtube上看一场单口相声。
2,test_00003,莱尼决定购买第一件或第三件裤子，他更注重质量而不是款式多样性。鲍勃建议他选择第一件，因为那样...,莱尼在鲍勃建议优先选择搭配性更好的款式后，决定购买第一或第三条裤子，并同意以质量为首要考虑因素。,莱尼无法决定买哪条裤子。鲍勃就此给莱尼提了些建议。莱尼听了他的建议，选了质量最好的裤子。
3,test_00004,艾玛现在不饿，所以她不会做晚饭。不过她很快就会到家，到时候她会通知威尔。,艾玛今晚不饿，不想做饭，但很快就会到家，到家后会通知威尔。,艾玛很快就会回家，而且她会告诉威尔。
4,test_00005,简和奥利确认周五晚上6点一起吃午饭，这是在他们讨论了18号的派对以及关于一瓶失踪的威士忌的有...,简和奥利确认周五下午 6 点课后一起吃午饭，此前两人还讨论了 18 号的派对，并就奥利手机里...,简在华沙，她和奥利有个聚会。她把重要的日子忘了，本来他们周五会共进午餐。但是奥利无意间给简打...


In [34]:
# Cell 16: Check whether Agent 3 changed the initial Chinese summary

if not df.empty:
    df["agent3_changed_output"] = (
        df["agent2_initial_chinese_summary"].str.strip()
        != df["agent3_final_chinese_summary"].str.strip()
    )

    change_summary = df["agent3_changed_output"].value_counts()

    print("Did Agent 3 change the output?")
    print(change_summary)

    df[[
        "id",
        "agent3_changed_output",
        "agent2_initial_chinese_summary",
        "agent3_final_chinese_summary",
    ]]
else:
    print("No results found.")

Did Agent 3 change the output?
agent3_changed_output
True    5
Name: count, dtype: int64
